# Explaining one prediction

MichAl Academy, lesson 2.9.

Run each cell with **Shift+Enter**.

Lesson 2.8 got the model to say a case is 87% likely rather than just "yes". The
next question anybody asks is which part of the evidence made it 87%, and there
are two standard answers, SHAP and LIME.

This notebook does four things with them. It checks SHAP's arithmetic against a
closed form, so the numbers stop being magic. It checks the property the whole
method rests on. It duplicates a feature and watches the explanation change
while nothing about the world does. And it puts SHAP and LIME on the same
prediction, where they turn out to disagree in a way that is completely
predictable once you know what each one is answering.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import shap
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from lime.lime_tabular import LimeTabularExplainer

# Colab and Kaggle both ship shap. lime is a pip install away on Colab:
#     !pip install lime
SEED = 0
data = load_breast_cancer()
X, y = data.data, data.target
names = list(data.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
print(f"{len(y_train)} training rows, {len(y_test)} held back, {X.shape[1]} features")
print(f"class 1 is 'benign' and is {y_train.mean():.1%} of the training rows")


## 1. Check the arithmetic before trusting the picture

Both methods produce a number per feature per prediction, and the temptation is
to read the resulting bar chart and move on. Do not, yet.

On a **linear** model the Shapley value has a closed form. Each feature's
contribution is its coefficient times how far the value sits from the training
mean:

```
contribution of feature i = beta_i x (x_i - mean_i)
```

So the library's answer can be checked by arithmetic rather than trusted.


In [ ]:
scaler = StandardScaler().fit(X_train)
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)
linear = LogisticRegression(max_iter=5000).fit(Z_train, y_train)

beta = linear.coef_[0]
mean_z = Z_train.mean(axis=0)
by_hand = beta * (Z_test[0] - mean_z)

# The masker is the reference distribution the explanation is measured against.
# Give it every training row; the default is to subsample, and section 1b shows
# what that does.
full = shap.maskers.Independent(Z_train, max_samples=len(Z_train))
lib = shap.LinearExplainer(linear, full)(Z_test[0:1]).values[0]

print(f"largest disagreement between the library and the closed form: "
      f"{np.abs(lib - by_hand).max():.3e}")
print(f"\nbase value from the library: {float(np.ravel(shap.LinearExplainer(linear, full).expected_value)[0]):.6f}")
print(f"beta . mean + intercept:     {beta @ mean_z + linear.intercept_[0]:.6f}")
print(f"\nbase + sum of contributions: {by_hand.sum() + beta @ mean_z + linear.intercept_[0]:.6f}")
print(f"the model's own log-odds:     {linear.decision_function(Z_test[0:1])[0]:.6f}")


Zero to the limit of floating point. On a linear model SHAP is not doing
anything clever, it is doing that multiplication, and you can check any bar in
any chart it draws with a calculator.

The last two lines are the property everything rests on. Lundberg and Lee's
paper calls the class of methods **additive feature attribution**, and this is
what additive means: the base value plus every contribution equals the model's
actual output for this case. Not approximately. Exactly.


## 1b. What is the explanation measured against?

Notice what had to be supplied above: a reference distribution. "How far the
value sits from the mean" needs a mean, and the answer changes with it.

The library will pick one for you. By default it subsamples the background to
100 rows.


In [ ]:
for label, n in (("subsampled to 100 (the default)", 100),
                 ("all 398 training rows", len(Z_train))):
    ex = shap.LinearExplainer(linear, shap.maskers.Independent(Z_train, max_samples=n))
    v = ex(Z_test[0:1]).values[0]
    print(f"{label:<34} base {float(np.ravel(ex.expected_value)[0]):>9.6f}"
          f"   largest gap from the closed form {np.abs(v - by_hand).max():.3e}")


The default moved the base value by 0.43 and every contribution by up to 0.11,
which on a log-odds scale is not a rounding error.

So an explanation is never "why the model said this". It is always **why the
model said this rather than what it says about some reference group**, and if
you did not choose the reference group, a default chose it. When you are asked
to justify a decision to somebody, that is the first question to be able to
answer.


## 2. The same check on a model with no closed form

A random forest has no formula to compare against, so the additive property is
the only thing available to verify. Check it on every held-back case at once.


In [ ]:
forest = RandomForestClassifier(random_state=SEED, n_jobs=1).fit(X_train, y_train)
tree_ex = shap.TreeExplainer(forest)

def shap_for(explainer, rows):
    """TreeExplainer returns one matrix per class for a classifier. Take the
    positive class and nothing else."""
    v = explainer.shap_values(rows)
    return v[..., 1] if v.ndim == 3 else v

sv = shap_for(tree_ex, X_test)
base = float(np.ravel(tree_ex.expected_value)[-1])
proba = forest.predict_proba(X_test)[:, 1]

recon = sv.sum(axis=1) + base
print(f"largest |base + contributions - model output| over {len(X_test)} cases:"
      f" {np.abs(recon - proba).max():.3e}")
print(f"base value {base:.4f}   training rate of class 1 {y_train.mean():.4f}")


Exact again, and the base value is the training rate of the positive class,
which is what "the average prediction" means here.

That is the whole mental model for SHAP: **it splits the distance between this
prediction and the average prediction across the features.** A case scoring
above the average has contributions that sum to a positive number, by
construction.

Hold on to that sentence. Section 4 is about what happens when you forget it.


In [ ]:
# One case, read as a list rather than a chart.
k = 40
order = np.argsort(-np.abs(sv[k]))[:8]
print(f"case {k}: the forest says {proba[k]:.4f}, truth is "
      f"{'benign' if y_test[k] else 'malignant'}")
print(f"  base value {base:+.4f}")
running = base
for j in order:
    running += sv[k][j]
    print(f"  {names[j]:<24} = {X_test[k][j]:>8.2f}   {sv[k][j]:+.4f}"
          f"   running total {running:.4f}")
rest = sv[k].sum() - sv[k][order].sum()
print(f"  {'the other 22 features':<24}                {rest:+.4f}"
      f"   running total {running + rest:.4f}")
print(f"  the model's actual output:                            {proba[k]:.4f}")


## 3. Duplicate a feature and watch the explanation move

Here is the failure that matters, because it looks like nothing went wrong.

Copy one column, so the pair now carries identical information, and refit. The
world is unchanged. The information available to the model is unchanged. See
what the explanation says.


In [ ]:
dup = names.index("worst radius")
Xd_train = np.hstack([X_train, X_train[:, [dup]]])
Xd_test = np.hstack([X_test, X_test[:, [dup]]])

forest_d = RandomForestClassifier(random_state=SEED, n_jobs=1).fit(Xd_train, y_train)
print(f"accuracy before {forest.score(X_test, y_test):.4f}"
      f"   after {forest_d.score(Xd_test, y_test):.4f}")

before = np.abs(shap_for(tree_ex, X_test[:100])).mean(axis=0)
after = np.abs(shap_for(shap.TreeExplainer(forest_d), Xd_test[:100])).mean(axis=0)

print(f"\nmean |contribution| for 'worst radius':")
print(f"  before duplication              {before[dup]:.4f}")
print(f"  after, the original             {after[dup]:.4f}")
print(f"  after, the copy                 {after[-1]:.4f}")
print(f"  after, the two together         {after[dup] + after[-1]:.4f}")
print(f"\ntotal across all features: before {before.sum():.4f}  after {after.sum():.4f}")

print("\nwhich features lost the credit:")
for j in np.argsort(after[:len(names)] - before)[:5]:
    print(f"  {names[j]:<24} {before[j]:.4f} -> {after[j]:.4f}"
          f"  ({after[j] - before[j]:+.4f})")


Read that in order.

The model got **worse**, 0.9532 to 0.9415, because a redundant column is not
free.

And the apparent importance of radius nearly **doubled**, 0.0599 to 0.1140.

The total across all features barely moved, 0.5318 to 0.5201, because the
additive property will not let it. So the extra credit had to come from
somewhere, and it came from `worst perimeter`, which halved from 0.0886 to
0.0439, along with `mean area` and `mean perimeter`. Every one of those is a
size measurement, correlated with radius.

Nothing here is a bug in SHAP. The attributions are a correct account of what
the fitted model does. But the fitted model changed because the feature list
changed, and the explanation reports the model. So the rule to carry is blunt:
**an explanation is a fact about your model, never a fact about the world.**
Read it as "radius matters twice as much" and you have made a claim about
tumours from a duplicated spreadsheet column.


## 4. SHAP and LIME on the same prediction

LIME takes a different route. Ribeiro, Singh and Guestrin describe it as
explaining any classifier "by learning an interpretable model locally around the
prediction": perturb the case, ask the model about the perturbations, fit a
small linear model to the answers, and report its coefficients.

Two methods, one prediction. Run both.


In [ ]:
lime_ex = LimeTabularExplainer(
    X_train, feature_names=names, class_names=["malignant", "benign"],
    discretize_continuous=False, random_state=SEED,
)

def lime_weights(idx, num_samples=4000):
    e = lime_ex.explain_instance(X_test[idx], forest.predict_proba,
                                 num_features=len(names), num_samples=num_samples)
    w = np.zeros(len(names))
    for j, val in e.as_map()[1]:
        w[j] = val
    return w, e.score

for k in (6, 151):
    w, r2 = lime_weights(k)
    st = int(np.argmax(np.abs(sv[k])))
    lt = int(np.argmax(np.abs(w)))
    print(f"\ncase {k}: forest says {proba[k]:.4f},"
          f" truth {'benign' if y_test[k] else 'malignant'}"
          f"   (LIME's local fit R2 {r2:.3f})")
    print(f"  SHAP's top feature: {names[st]:<20} {sv[k][st]:+.4f}")
    print(f"  LIME's top feature: {names[lt]:<20} {w[lt]:+.4f}")
    print(f"  and on {names[st]}: SHAP {sv[k][st]:+.4f}, LIME {w[st]:+.4f}")


Both methods pick the same top feature on both cases. On case 6 they agree on
its sign. On case 151 they are **opposite**.

The obvious suspicion is that case 151 broke LIME, since the forest scores it at
exactly 1.0000 and a local sample around a saturated prediction might have
nothing to fit. Ruled out: LIME's local R-squared is 0.695 there, against 0.698 on case 6.

So find out which one is wrong by asking the model directly. Case 151's worst
perimeter is 82.0, well below the training mean of 107.0. Push it up.


In [ ]:
j = names.index("worst perimeter")
k = 151
print(f"case {k}, worst perimeter is {X_test[k][j]:.1f}"
      f" (training mean {X_train[:, j].mean():.1f})")
for mult in (0.8, 1.0, 1.2, 1.5, 2.0):
    xq = X_test[k].copy()
    xq[j] = X_test[k][j] * mult
    print(f"  set it to {xq[j]:6.1f}   the model says {forest.predict_proba([xq])[0, 1]:.4f}")


Increasing worst perimeter drives the benign probability **down**, from 1.0000
to 0.7900. So LIME's negative weight is correct.

And SHAP's positive value is also correct. This case scores 1.0000 against a
base value of 0.6289, so its contributions must sum to +0.371, and a small
perimeter is exactly why it sits above average.

Neither method is wrong. They answer different questions:

- **SHAP:** why is this prediction different from the average prediction?
- **LIME:** which way would this prediction move if I changed this feature?

For a case *below* the base value those two answers point the same way, and
above it they generally do not. Which is testable, so test it.


In [ ]:
above = {"same": 0, "opposite": 0}
below = {"same": 0, "opposite": 0}
for k in range(0, len(X_test), 2):        # every other case, to keep this under a minute
    w, _ = lime_weights(k, num_samples=1500)
    top = int(np.argmax(np.abs(sv[k])))
    if w[top] == 0:
        continue
    key = "same" if np.sign(sv[k][top]) == np.sign(w[top]) else "opposite"
    (above if proba[k] > base else below)[key] += 1

for label, box in (("prediction above the base value", above),
                   ("prediction below the base value", below)):
    total = box["same"] + box["opposite"]
    print(f"{label}: {total} cases, signs agree on {box['same']} ({box['same'] / total:.0%})")


100% below the base value and 6% above it.

That is not two methods being unreliable. It is two methods answering two
questions, and the sign clash appearing exactly where the questions come apart.

Which one do you want? It depends on what the person asking needs:

**"Why was this flagged?"** wants SHAP. It is an account of this case against
the population, and it adds up to the decision.

**"What would have to change for this not to be flagged?"** wants a local slope,
so LIME, or better, a counterfactual method built for that question.

Handing over the wrong one, or worse, handing over both and calling the
disagreement noise, is how explanations lose the trust they were supposed to
build.


## 5. The argument for not needing any of this

Rudin's 2019 paper in Nature Machine Intelligence puts the opposing case in its
title: *Stop Explaining Black Box Machine Learning Models for High Stakes
Decisions and Use Interpretable Models Instead*. Her argument is that "trying
to explain black box models, rather than creating models that are interpretable
in the first place, is likely to perpetuate bad practices and can potentially
cause catastrophic harm to society."

The measurement in section 3 is a small piece of evidence for her. A duplicated
column changed the explanation's headline and nothing else, and no amount of
care reading the chart would have caught it.

There is also a cheap test available. Lesson 2.6 measured a random forest
beating boosting on two of three datasets, and lesson 2.4's logistic regression
is directly readable from its coefficients. Before reaching for an explanation
method, check what a model you can read outright actually costs you.


In [ ]:
simple = LogisticRegression(max_iter=5000).fit(Z_train, y_train)
print(f"logistic regression, readable by inspection: {simple.score(Z_test, y_test):.4f}")
print(f"random forest, needs SHAP:                   {forest.score(X_test, y_test):.4f}")
print("\nthe five largest coefficients, which is the entire explanation:")
for j in np.argsort(-np.abs(simple.coef_[0]))[:5]:
    print(f"  {names[j]:<24} {simple.coef_[0][j]:+.4f}")


On this dataset the readable model **wins**, 0.9591 to 0.9532, and its entire
explanation is five numbers you can print.

That is not an argument against SHAP. It is an argument for measuring the price
of the black box before you agree to pay it, because here the price was
negative.


## What to take from this

| Claim | What we measured |
|---|---|
| SHAP values are a black box themselves | No. On a linear model they equal beta x (x - mean) to floating-point zero |
| The explanation adds up to the prediction | Yes, exactly. Largest error over 171 cases was 4e-16 |
| An explanation tells you what matters in the world | No. Duplicating a column doubled radius's apparent importance and made the model worse |
| SHAP and LIME agree | Below the base value 100% of the time, above it 6% |
| A sign clash means one method is broken | No. Both were verified correct against the model. They answer different questions |
| You need an explanation method | Not always. Logistic regression beat the forest here, 0.9591 to 0.9532, and is readable outright |


## Try this

1. Duplicate a different feature. Does the credit still get taken from the
   features correlated with it, or does the pattern depend on which one you
   pick?
2. Set `discretize_continuous=True` in the LIME explainer, which is its
   default. The weights change meaning entirely, since they now describe a bin
   rather than a slope. Does the sign clash survive?
3. Run `shap.TreeExplainer` on the boosted model from lesson 2.6 and compare
   the top features against the forest's on the same case. Two models fitted on
   identical data with similar accuracy, and an analyst reading the two
   explanations would write two different incident reports.
